<a href="https://colab.research.google.com/github/HARIHARAN-0007/sfm-camera-calibration/blob/main/Camera%20Calibration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install opencv-python numpy

In [ ]:
import os

os.makedirs("/content/calibration", exist_ok=True)

print("Calibration folder created successfully!")

Calibration folder created successfully!


In [ ]:
from google.colab import files
import os

folder = "/content/calibration"
os.makedirs(folder, exist_ok=True)

uploaded = files.upload()

for filename, data in uploaded.items():
    with open(os.path.join(folder, filename), "wb") as f:
        f.write(data)

print("Photos saved:", len(uploaded))
print("Folder location:", folder)

Saving cali12.JPG to cali12.JPG
Saving cali11.JPG to cali11.JPG
Saving cali10.JPG to cali10.JPG
Saving cali9.JPG to cali9.JPG
Saving cali8.JPG to cali8.JPG
Saving cali7.JPG to cali7.JPG
Saving cali6.JPG to cali6.JPG
Saving cali5.JPG to cali5.JPG
Saving cali4.JPG to cali4.JPG
Saving cali3.JPG to cali3.JPG
Saving cali2.JPG to cali2.JPG
Saving cali1.JPG to cali1.JPG
Photos saved: 12
Folder location: /content/calibration


In [ ]:
import os
import glob
import json
import cv2
import numpy as np

# ======================================================
# CONFIGURATION
# ======================================================

CALIBRATION_FOLDER = "/content/calibration"

# 11 x 10 squares => 10 x 9 inner corners
CHECKERBOARD = (10, 9)

# Your checkerboard square size
SQUARE_SIZE_MM = 25.0

# Resize large images before corner detection
MAX_IMAGE_DIMENSION = 1600

CRITERIA = (
    cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_MAX_ITER,
    30,
    0.001
)

FAST_FLAGS = (
    cv2.CALIB_CB_ADAPTIVE_THRESH
    | cv2.CALIB_CB_NORMALIZE_IMAGE
    | cv2.CALIB_CB_FAST_CHECK
)

NORMAL_FLAGS = (
    cv2.CALIB_CB_ADAPTIVE_THRESH
    | cv2.CALIB_CB_NORMALIZE_IMAGE
)

# ======================================================
# LOAD IMAGES
# ======================================================

image_paths = sorted(
    p for p in glob.glob(os.path.join(CALIBRATION_FOLDER, "*"))
    if p.lower().endswith(
        (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
    )
)

if not image_paths:
    raise FileNotFoundError(
        "No photos found in /content/calibration. "
        "Upload your chessboard photos first."
    )

print("Images found:", len(image_paths))
print("Expected inner corners:", CHECKERBOARD)

# ======================================================
# PREPARE CHECKERBOARD 3D POINTS
# ======================================================

cols, rows = CHECKERBOARD

object_template = np.zeros(
    (cols * rows, 3),
    dtype=np.float32
)

object_template[:, :2] = (
    np.mgrid[0:cols, 0:rows].T.reshape(-1, 2)
    * SQUARE_SIZE_MM
)

object_points = []
image_points = []
accepted_images = []
rejected_images = []
image_size = None

# ======================================================
# DETECT CORNERS
# ======================================================

for i, path in enumerate(image_paths, start=1):

    filename = os.path.basename(path)
    print(f"\n[{i}/{len(image_paths)}] Checking {filename}")

    image = cv2.imread(path)

    if image is None:
        print("  Image could not be read.")
        rejected_images.append(filename)
        continue

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    current_size = (gray.shape[1], gray.shape[0])

    if image_size is None:
        image_size = current_size

    elif current_size != image_size:
        print("  Skipped: image resolution differs.")
        rejected_images.append(filename)
        continue

    # Resize large images to make detection faster.
    h, w = gray.shape[:2]
    scale = min(1.0, MAX_IMAGE_DIMENSION / max(h, w))

    if scale < 1.0:
        small = cv2.resize(
            gray,
            None,
            fx=scale,
            fy=scale,
            interpolation=cv2.INTER_AREA
        )
    else:
        small = gray

    # First try the fast detector.
    found, corners = cv2.findChessboardCorners(
        small,
        CHECKERBOARD,
        FAST_FLAGS
    )

    # If needed, try again without the fast rejection check.
    if not found:
        found, corners = cv2.findChessboardCorners(
            small,
            CHECKERBOARD,
            NORMAL_FLAGS
        )

    # Fall back to OpenCV's more robust detector if available.
    if not found and hasattr(cv2, "findChessboardCornersSB"):
        found, corners = cv2.findChessboardCornersSB(
            small,
            CHECKERBOARD,
            cv2.CALIB_CB_NORMALIZE_IMAGE
        )

    if not found:
        print("  NOT DETECTED")
        rejected_images.append(filename)
        continue

    # Convert detected coordinates back to original image scale.
    corners = corners.astype(np.float32) / scale

    # Refine using the full-resolution image.
    corners = cv2.cornerSubPix(
        gray,
        corners,
        (11, 11),
        (-1, -1),
        CRITERIA
    )

    object_points.append(object_template.copy())
    image_points.append(corners)
    accepted_images.append(filename)

    print("  DETECTED successfully")

print("\n========== DETECTION SUMMARY ==========")
print("Total photos:", len(image_paths))
print("Successful:", len(accepted_images))
print("Failed or skipped:", len(rejected_images))

if len(accepted_images) < 3:
    raise RuntimeError(
        "Calibration stopped because fewer than 3 images were detected. "
        "Check the actual checkerboard pattern and inner-corner count."
    )

# ======================================================
# CALIBRATE CAMERA
# ======================================================

print("\nCalculating camera parameters...")

rms, K, distortion, rvecs, tvecs = cv2.calibrateCamera(
    object_points,
    image_points,
    image_size,
    None,
    None
)

# ======================================================
# CALCULATE REPROJECTION RMSE
# ======================================================

sum_squared_error = 0.0
total_points = 0

for obj, observed, rvec, tvec in zip(
    object_points, image_points, rvecs, tvecs
):
    projected, _ = cv2.projectPoints(
        obj, rvec, tvec, K, distortion
    )

    differences = (
        observed.reshape(-1, 2)
        - projected.reshape(-1, 2)
    )

    sum_squared_error += float(np.sum(differences ** 2))
    total_points += len(differences)

reprojection_rmse = np.sqrt(
    sum_squared_error / total_points
)

# ======================================================
# DISPLAY RESULTS
# ======================================================

print("\n========== CAMERA CALIBRATION RESULTS ==========")
print("Image resolution:", image_size)

print("\nCamera matrix K:")
print(K)

print("\nDistortion coefficients:")
print(distortion)

print(f"\nCalibration RMS: {rms:.6f} pixels")
print(f"Reprojection RMSE: {reprojection_rmse:.6f} pixels")

if rms < 1.0 and reprojection_rmse < 1.0:
    print("\nInitial error check: PASSED")
else:
    print("\nInitial error check: REVIEW REQUIRED")

# ======================================================
# SAVE RESULTS
# ======================================================

output_folder = "/content/calibration_output"
os.makedirs(output_folder, exist_ok=True)

results = {
    "checkerboard_inner_corners": list(CHECKERBOARD),
    "square_size_mm": SQUARE_SIZE_MM,
    "image_resolution": list(image_size),
    "accepted_images": accepted_images,
    "rejected_images": rejected_images,
    "camera_matrix": K.tolist(),
    "distortion_coefficients": distortion.tolist(),
    "calibration_rms_pixels": float(rms),
    "reprojection_rmse_pixels": float(reprojection_rmse)
}

json_path = os.path.join(
    output_folder, "camera_calibration.json"
)

npz_path = os.path.join(
    output_folder, "camera_calibration.npz"
)

with open(json_path, "w") as f:
    json.dump(results, f, indent=4)

np.savez(
    npz_path,
    camera_matrix=K,
    distortion_coefficients=distortion,
    image_size=np.array(image_size)
)

print("\nCalibration results saved:")
print(json_path)
print(npz_path)

Images found: 24
Expected inner corners: (10, 9)

[1/24] Checking IMG_2024.jpeg
  DETECTED successfully

[2/24] Checking IMG_2025.jpeg
  DETECTED successfully

[3/24] Checking IMG_2026.jpeg
  DETECTED successfully

[4/24] Checking IMG_2027.jpeg
  DETECTED successfully

[5/24] Checking IMG_2028.jpeg
  DETECTED successfully

[6/24] Checking IMG_2029.jpeg
  DETECTED successfully

[7/24] Checking IMG_2030.jpeg
  DETECTED successfully

[8/24] Checking IMG_2031.jpeg
  DETECTED successfully

[9/24] Checking IMG_2032.jpeg
  DETECTED successfully

[10/24] Checking IMG_2033.jpeg
  DETECTED successfully

[11/24] Checking IMG_2034.jpeg
  DETECTED successfully

[12/24] Checking IMG_2035.jpeg
  DETECTED successfully

[13/24] Checking cali1.JPG
  Skipped: image resolution differs.

[14/24] Checking cali10.JPG
  Skipped: image resolution differs.

[15/24] Checking cali11.JPG
  Skipped: image resolution differs.

[16/24] Checking cali12.JPG
  Skipped: image resolution differs.

[17/24] Checking cali2.J